<a href="https://colab.research.google.com/github/FreddieLewin23/FreddieLewin23/blob/main/SIG_data_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/Data Exercise - Trade Data.csv')
df.head() # closed trade pool for 7 traders, 120,000 rows and spot
# data looks very tidy, no missing data, dates are in the correct format
df['Date'] = pd.to_datetime(df['Date'])
df.isna().sum()


,0
Date,0
TimeOfDay,0
Spot,0
BuyTrader,0
SellTrader,0


Assumptions of the data:
1. I start trading as soon as the data ends here. The strategy needs to be based off of what I have in the data without overfitting (hopefully)
2. Spot price has no pattern (ie Brownian motion etc) so strategy needs to be based off of the volume data only.
3. In live trading I will not be able to see the orders of who is trading (only see at the end of the day). If I do enter into a trade then I can see who I entered into the trade with. This will rule out intra-day strategies that rely on order flow from traders that seem profitbale in the training data. When I run backtesting on strategies I need to assume I can only see spot, time and the people I enter into trades with.

Constraints of trading:
1. up to 5 buys and 5 sells (so can do less) and need to end each day flat. Short selling allowed.
2. No trades last 30 minutes of trading (weird one here). If not flat going into last 30 minutes, I need to enter into the first trades that come up towards EoD.

Ideas and Plan of Action:
Firstly I want to have some idea of how the other traders trade with some simple analysis.

1. You can’t target flows by ID unless you “poke” the tape and see the counterparty. So strategies must either (i) probe to reveal IDs, or (ii) time-target windows where good traders are likely.
2.

In [14]:
df["Date"] = pd.to_datetime(df["Date"])
try:
    df["TimeOfDay"] = pd.to_datetime(df["TimeOfDay"]).dt.time
except Exception:
    pass
dt = pd.to_datetime(df["Date"].astype(str) + " " + df["TimeOfDay"].astype(str))
df["dt"] = dt
df = df.sort_values("dt").reset_index(drop=True)
price_col = None
for cand in ["Spot", "Price", "spot", "price", "Spot_Price"]:
    if cand in df.columns:
        price_col = cand
        break
buy_col = None
sell_col = None
for cand in df.columns:
    if "buy" in cand.lower():
        buy_col = cand
    if "sell" in cand.lower():
        sell_col = cand

# Derive day, time, and enforcement windows
df["day"] = df["dt"].dt.date
df["tod"] = df["dt"].dt.time

# Infer trading hours window (first and last time-of-day per day)
agg = df.groupby("day")["tod"].agg(["min", "max"]).reset_index()
start_t = agg["min"].mode().iloc[0]
end_t = agg["max"].mode().iloc[0]

cutoff_minutes = 30
# Compute cutoff time per day based on per-day max time
# We'll approximate by subtracting 30 minutes from each day's max and then take the mode
per_day_cutoff = (df.groupby("day")["dt"].max() - pd.Timedelta(minutes=cutoff_minutes)).dt.time
cutoff_t = per_day_cutoff.mode().iloc[0]


In [19]:
per_trader = {}
if buy_col and sell_col:
    # All trader IDs that appear anywhere
    traders = pd.unique(pd.concat([df[buy_col], df[sell_col]], ignore_index=True))
    traders = np.sort(traders)
    # Trades where each trader appears on either side
    rows = []
    for tr in traders:
        buys = (df[buy_col] == tr).sum()
        sells = (df[sell_col] == tr).sum()
        total = buys + sells
        # Average time-of-day (in seconds) to see if someone is early/late concentrated
        sec = df.loc[(df[buy_col] == tr) | (df[sell_col] == tr), "dt"].dt.hour * 3600 + df.loc[(df[buy_col] == tr) | (df[sell_col] == tr), "dt"].dt.minute * 60
        avg_time_sec = sec.mean() if len(sec) else np.nan
        rows.append({"Trader": tr, "Buys": buys, "Sells": sells, "Total": total, "AvgTimeSec": avg_time_sec})
    per_trader_df = pd.DataFrame(rows).sort_values("Total", ascending=False).reset_index(drop=True)
else:
    per_trader_df = pd.DataFrame()
per_trader_df

,Trader,Buys,Sells,Total,AvgTimeSec
0,0,46188,56763,102951,45844.403648
1,4,21113,21113,42226,45595.765642
2,2,17878,17878,35756,45367.623336
3,5,16906,16906,33812,45617.651130
4,1,7018,7018,14036,45365.407524
5,3,10570,0,10570,45532.836329
6,6,575,570,1145,36268.873362


Looks like Trader 5 is long only, maybe something like a pension fund or retail trader just wanting long term exposure, where as trader 1, 2, 3, 4, 6 are flat overall so are market makers. trader 0 is overall short so may be something like a hedge fund who are taking positions on the direction of spot.